
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQO_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model(
    "groq:openai/gpt-oss-20b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000229EEAA4C20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000229EEAA56A0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")
    

In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000229EEAA4C20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000229EEAA56A0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The titl

In [5]:
## how the input and output looks like

response=model_with_structure.invoke("Tell me about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside  parsed structure

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details about the movie "Inception". The user wants details. We have a function "Movie" that takes director, rating, title, year. We can call that function. The user didn\'t specify rating but we can provide. The function expects these arguments: director: string, rating: number, title: string, year: number. We can supply rating 8.8 (approx). So we call the function.', 'tool_calls': [{'id': 'fc_81e7b2e9-3412-4880-9af7-0e09b3aa20b2', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 164, 'total_tokens': 293, 'completion_time': 0.134577498, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0.008003067, 'prompt_tokens_details': None, 'queue_time': 0.311978431, 'total_time': 0.142580565}, 'model_name': 'opena

## Nested Structure

In [8]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genre:str
    budget:float | None = Field(default=None, description="The budget of the movie in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")

response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito')], genre='Sci-Fi/Thriller', budget=160000000.0)

# TypedDict

TypedDict provides a simpler alternative using Python's built in typing,ideal when you don't need runtime validation

In [10]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    cast: Annotated[list[str], "The cast of the movie"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The rating of the movie out of 10"]

model_withtyped_dict = model.with_structured_output(MovieDict)
model_withtyped_dict.invoke("Please provide details about the movie Inception")
model_withtyped_dict.invoke("Please provide details about the movie Inception")

{'cast': ['Leonardo DiCaprio',
  'Joseph Gordon-Levitt',
  'Elliot Page',
  'Tom Hardy',
  'Ken Watanabe',
  'Cillian Murphy',
  'Marion Cotillard'],
 'director': 'Christopher Nolan',
 'rating': 8.8,
 'title': 'Inception',
 'year': 2010}

In [16]:
# model_withtyped_dict.profile()

'''Model kya support karta hai, jaise ki structured output, aur kaunse features 
available hain. Ye aapko model ke bare me aur jankari dega.'''

model.profile  # It will give you the profile of the model, including its capabilities and limitations.



{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [22]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash"
)

In [23]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result)

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='96178f7b-157a-4185-88f8-1b58e792de4f'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'ContactInfo', 'arguments': '{"email": "john@example.com", "phone": "(555) 123-4567", "name": "John Doe"}'}, '__gemini_function_call_thought_signatures__': {'call_1187301': 'EvECCu4CAWkUfRN0koBIbZ+WOnzVR+F5Cu5q1iT+45J0SdY1VRLWgVRgf1lu+dpzfrVgFPC9VO2/XZeexwtpPBjZtcsrehtXwTFwzBKgjyPHKv5DpgpJv3mktcfbsG/GVkpnX6mgOyM4de0cSCBRIdFZ6EoUCNM1LMXBPu6tizJGLYPXzwj1SVhJRabuX1s52tB6TwBBhOl90G8dLLaSvR6zD9OqnTDziT8oXhxV9MhgPTm9g2Pgagm9kkw63wsVJyOJgKE7MOi0boXSIP6nWSTx4qnZVo0Z9PTe6KAwN25yovJ/wr+91dXHJU7Au34IzdVxrngpzW43kOFXEUdu3yx6fm9Ao6Q0TQ9pwBi8bf4jLjP9WO9Ql0ouM6O+a++C+fsOr6lIpZT0y/g45339qNoTdYTysCrQp2FvI/CvN+eLF0MTdS7BvHlWl7oW6IC9SH+VH/dIWz+kZB20G9Azlanq4wUAkrxY3jCk3Gk/ud6U+ual'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gem

In [24]:
result["structured_response"]  # This will give you the structured response in the form of a ContactInfo object.

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [25]:
## Type Dict

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result["structured_response"])

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [26]:
## Data class

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent= create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
